# 남은 두 칸: GPU에서 돌리면 달라지는가

**목적.** 원인을 못 찾은 두 칸을 GPU에서 같은 설정으로 세 번씩 돌린다. CPU 결과와 같은지, GPU에서 실행마다 흔들리는지 본다. 저자는 GPU로 표를 만들었으므로, GPU 계산 차이가 원인인지 확인하는 실험이다.

| 칸 | 코드 | CPU 결과 | 논문 |
|---|---|---|---|
| 다변량 DLinear ETTm2 96 | 2022-06 (`04ffd8d`) | 0.16766 / 0.25980 | 0.167 / 0.260 |
| NLinear 환율 720 | 2022-08 (`7f7a25b`) | 0.94301 / 0.72828 | 1.033 / 0.780 |

**실행.** 런타임 **GPU → T4**(일반 RAM) → **모두 실행**. 10분 안팎. 끝나면 맨 아래 셀 출력을 채팅에 붙여 주세요.

In [ ]:
!nvidia-smi --query-gpu=name --format=csv
import os, subprocess
os.chdir('/content')
if not os.path.exists('LTSF-Linear'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/cure-lab/LTSF-Linear.git'], check=True)
os.chdir('/content/LTSF-Linear')
for name, commit in [('june', '04ffd8d'), ('aug', '7f7a25b')]:
    d = f'/content/{name}'
    if not os.path.exists(d):
        r = subprocess.run(['git', 'worktree', 'add', '-q', d, commit])
        if r.returncode != 0:
            subprocess.run(['git', 'fetch', '-q', '--unshallow'])
            subprocess.run(['git', 'worktree', 'add', '-q', d, commit], check=True)
    subprocess.run(f"sed -i 's/np\\.Inf/np.inf/' {d}/utils/tools.py", shell=True, check=True)
    os.makedirs(f'{d}/dataset', exist_ok=True)
if not os.path.exists('/content/raw_data'):
    subprocess.run('pip -q install gdown && gdown -q --folder https://drive.google.com/drive/folders/1ZOYpTUa82_jCcxIdTmyr0LXQfvaM9vIy -O /content/raw_data', shell=True, check=True)
for name in ['june', 'aug']:
    for f in ['ETTm2.csv', 'exchange_rate.csv']:
        subprocess.run(f'find /content/raw_data -name "{f}" -exec cp -n {{}} /content/{name}/dataset/ \;', shell=True)
!ls -la /content/june/dataset /content/aug/dataset
!md5sum /content/june/dataset/*.csv

In [ ]:
import re, time
RUNS = [
  ('DLinear ETTm2 96', 'june', '--data_path ETTm2.csv --data ETTm2 --features M --seq_len 336 --pred_len 96 --enc_in 7 --model DLinear --batch_size 32 --learning_rate 0.001'),
  ('NLinear Exchange 720', 'aug', '--data_path exchange_rate.csv --data custom --features M --seq_len 336 --pred_len 720 --enc_in 8 --model NLinear --batch_size 32 --learning_rate 0.0005'),
]
out = []
for cell, d, args in RUNS:
    for rep in range(3):
        t = time.time()
        cmd = f'cd /content/{d} && python -u run_longExp.py --is_training 1 --root_path ./dataset/ --model_id gpu{rep} --des Exp --itr 1 {args}'
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        m = re.search(r'^mse:([0-9.]+), mae:([0-9.]+)', r.stdout, re.M)
        out.append((cell, rep + 1, m.group(1) if m else 'FAIL', m.group(2) if m else (r.stdout + r.stderr)[-300:]))
        print(cell, rep + 1, out[-1][2], out[-1][3], f'{time.time() - t:.0f}s')
        subprocess.run(f'rm -rf /content/{d}/checkpoints', shell=True)

## 결과를 채팅에 붙여 주세요

In [ ]:
import torch
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'cudnn', torch.backends.cudnn.version())
for o in out: print(','.join(map(str, o)))